# 04. Dome calibrations pipeline

### Imports

In [ ]:
from pathlib import Path
import ast
import re
import numpy as np
import pandas as pd

## Dome DIRECT compass calibrations

## First, create a mapping file.................

In [ ]:
# ---------------------------------------------------------------
# Paths
# ---------------------------------------------------------------

# Your filled mapping file
DOME_DIRECT_MAPPING_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome_north_calibrations\dome_direct_calibrations\dome_direct_calibration_MAPPING_FILLED.csv"
)

# Folder containing all actual dome waggle dance annotation CSVs.
# This can be a high-level folder; the script searches subfolders.
DOME_DANCE_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome"
)

# Folder containing all dome compass calibration files
DOME_COMPASS_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome_north_calibrations\dome_direct_calibrations\dome_compass_calibrations"
)

In [ ]:
def parse_annotation_list(value, column_name=""):
    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value).strip()

    value = re.sub(
        r"np\.float(?:16|32|64)?\(([^)]+)\)",
        r"\1",
        value
    )

    value = re.sub(r"\bnan\b", "None", value, flags=re.IGNORECASE)

    # Repair common missing opening bracket problem:
    # "(1, 2), (3, 4)]" -> "[(1, 2), (3, 4)]"
    if value.startswith("(") and value.endswith("]"):
        value = "[" + value

    try:
        return list(ast.literal_eval(value))
    except (ValueError, SyntaxError) as error:
        raise ValueError(
            f"Could not parse column '{column_name}': {value}"
        ) from error


def vector_to_video_angle_deg(u, v):
    """
    Video convention:
    0°   = top/up in video
    90°  = right
    180° = bottom/down
    270° = left
    """
    return np.degrees(np.arctan2(u, -v)) % 360


def circular_mean_deg(angles_deg):
    angles_rad = np.radians(angles_deg)

    mean_rad = np.arctan2(
        np.mean(np.sin(angles_rad)),
        np.mean(np.cos(angles_rad))
    )

    return np.degrees(mean_rad) % 360


def circular_resultant_length_deg(angles_deg):
    angles_rad = np.radians(angles_deg)

    return np.sqrt(
        np.mean(np.cos(angles_rad))**2
        + np.mean(np.sin(angles_rad))**2
    )


def circular_sd_deg(angles_deg):
    R = circular_resultant_length_deg(angles_deg)
    R = np.clip(R, 1e-12, 1)

    return np.degrees(np.sqrt(-2 * np.log(R)))


def clean_filename_value(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip().replace('"', "").replace("'", "")

    if x == "" or x.lower() == "nan":
        return pd.NA

    return x


def clean_calibration_id(x):
    """
    Converts:
        21_calibration_11-21-50_waggle_annotations.csv
        21_calibration_11-21-50_waggle_annotations
        C:/.../21_calibration_11-21-50_waggle_annotations.csv

    into:
        21_calibration_11-21-50
    """
    if pd.isna(x):
        return pd.NA

    x = str(x).strip().replace('"', "").replace("'", "")
    x = Path(x).name

    if x.endswith(".csv"):
        x = x[:-4]

    if x.endswith("_waggle_annotations"):
        x = x.replace("_waggle_annotations", "")

    return x


def strip_annotation_suffix(filename):
    name = Path(filename).name.strip()

    if name.endswith(".csv"):
        name = name[:-4]

    if name.endswith("_waggle_annotations"):
        name = name.replace("_waggle_annotations", "")

    return name


def resolve_file(file_value, search_roots, label="file"):
    """
    Allows mapping file entries as:
    - full path
    - relative path from root
    - filename with .csv
    - filename without .csv
    """
    value = clean_filename_value(file_value)

    if pd.isna(value):
        raise ValueError(f"Missing {label} in mapping file.")

    value = str(value)

    value_path = Path(value)

    # 1. Full path
    if value_path.exists():
        return value_path

    possible_values = [value]

    if not value.endswith(".csv"):
        possible_values.append(value + ".csv")

    # 2. Relative path from search root
    for root in search_roots:
        root = Path(root)

        for possible_value in possible_values:
            candidate = root / possible_value

            if candidate.exists():
                return candidate

    # 3. Recursive filename search
    all_matches = []

    for root in search_roots:
        root = Path(root)

        root_matches = []

        for possible_value in possible_values:
            possible_name = Path(possible_value).name
            root_matches.extend(root.rglob(possible_name))

        root_matches = sorted(set(root_matches))
        all_matches.extend(root_matches)

        if len(root_matches) == 1:
            return root_matches[0]

        if len(root_matches) > 1:
            raise ValueError(
                f"Found multiple matches for {label}: {value}\n"
                "Use a relative path in the mapping file to disambiguate.\n\n"
                "Matches found:\n"
                + "\n".join(str(match) for match in root_matches)
            )

    raise FileNotFoundError(
        f"Could not find {label}: {value}\n"
        "Searched in:\n"
        + "\n".join(str(root) for root in search_roots)
    )

In [ ]:
mapping_df = pd.read_csv(
    DOME_DIRECT_MAPPING_FILE,
    dtype=str,
    encoding="utf-8-sig"
)

mapping_df.columns = mapping_df.columns.str.strip()

required_columns = [
    "date",
    "bee_id",
    "time",
    "dance_file",
    "calibration_file",
    "condition"
]

missing_columns = [
    col for col in required_columns
    if col not in mapping_df.columns
]

if missing_columns:
    raise ValueError(
        "Your mapping file is missing these columns:\n"
        + "\n".join(missing_columns)
        + "\n\nColumns found were:\n"
        + "\n".join(repr(col) for col in mapping_df.columns)
    )

# Clean empty cells
mapping_df = mapping_df.replace(r"^\s*$", pd.NA, regex=True)

# Remove fully empty rows
mapping_df = mapping_df.dropna(
    subset=["dance_file", "calibration_file"],
    how="all"
).reset_index(drop=True)

# Clean filenames
mapping_df["dance_file"] = mapping_df["dance_file"].apply(clean_filename_value)
mapping_df["calibration_file"] = mapping_df["calibration_file"].apply(clean_filename_value)
mapping_df["calibration_id"] = mapping_df["calibration_file"].apply(clean_calibration_id)

print(f"Mapping rows loaded: {len(mapping_df)}")
display(mapping_df.head())

In [ ]:
check_rows = []

for idx, row in mapping_df.iterrows():
    try:
        dance_path = resolve_file(
            row["dance_file"],
            search_roots=[DOME_DANCE_ROOT],
            label="dance_file"
        )

        dance_error = ""
        n_dance_matches = 1

    except Exception as error:
        dance_path = ""
        dance_error = str(error)
        n_dance_matches = 0

    try:
        calibration_path = resolve_file(
            row["calibration_file"],
            search_roots=[DOME_COMPASS_ROOT],
            label="calibration_file"
        )

        calibration_error = ""
        n_calibration_matches = 1

    except Exception as error:
        calibration_path = ""
        calibration_error = str(error)
        n_calibration_matches = 0

    check_rows.append({
        "row_number": idx + 1,
        "date": row.get("date", ""),
        "bee_id": row.get("bee_id", ""),
        "time": row.get("time", ""),
        "dance_file": row["dance_file"],
        "dance_path": str(dance_path),
        "dance_error": dance_error,
        "calibration_file": row["calibration_file"],
        "calibration_path": str(calibration_path),
        "calibration_error": calibration_error
    })

file_check_df = pd.DataFrame(check_rows)

file_check_df.to_csv(OUTPUT_FILE_CHECK, index=False)

problem_rows = file_check_df[
    (file_check_df["dance_error"] != "")
    | (file_check_df["calibration_error"] != "")
]

print(f"Rows checked: {len(file_check_df)}")
print(f"Problem rows: {len(problem_rows)}")

display(problem_rows)

In [ ]:
def extract_direction_angles_from_csv(csv_file):
    """
    Used for compass calibration files.
    Only direction vectors are needed.
    """
    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    for session_index, row in df.iterrows():
        directions = parse_annotation_list(
            row["waggle_directions"],
            "waggle_directions"
        )

        try:
            frames = parse_annotation_list(
                row["waggle_start_frames"],
                "waggle_start_frames"
            )
        except Exception:
            frames = []

        for i, direction in enumerate(directions, start=1):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            # Skip invalid 0,0 vectors
            if np.hypot(u, v) < 1e-8:
                continue

            video_angle_deg = vector_to_video_angle_deg(u, v)

            frame = frames[i - 1] if i - 1 < len(frames) else np.nan

            records.append({
                "calibration_file": csv_file.name,
                "calibration_id": clean_calibration_id(csv_file.name),
                "annotation_number": len(records) + 1,
                "original_annotation_index": i,
                "frame": frame,
                "direction_u": u,
                "direction_v": v,
                "calibration_video_angle_deg": video_angle_deg,
                "annotation_session": session_index + 1
            })

    return pd.DataFrame(records)


calibration_rows = []
calibration_raw_dfs = []

for calibration_file in mapping_df["calibration_file"].dropna().drop_duplicates():
    calibration_path = resolve_file(
        calibration_file,
        search_roots=[DOME_COMPASS_ROOT],
        label="calibration_file"
    )

    cal_raw = extract_direction_angles_from_csv(calibration_path)

    if len(cal_raw) == 0:
        raise ValueError(f"No valid compass annotations found in {calibration_path.name}")

    calibration_raw_dfs.append(cal_raw)

    magnetic_north_deg = circular_mean_deg(
        cal_raw["calibration_video_angle_deg"].values
    )

    true_north_video_deg = (
        magnetic_north_deg - DECLINATION_DEG
    ) % 360

    calibration_rows.append({
        "calibration_file": calibration_path.name,
        "calibration_id": clean_calibration_id(calibration_path.name),
        "calibration_n_annotations": len(cal_raw),
        "calibration_mean_north_deg": magnetic_north_deg,
        "calibration_circular_sd_deg": circular_sd_deg(
            cal_raw["calibration_video_angle_deg"].values
        ),
        "calibration_R": circular_resultant_length_deg(
            cal_raw["calibration_video_angle_deg"].values
        ),
        "declination_deg": DECLINATION_DEG,
        "true_north_video_deg": true_north_video_deg,
        "calibration_notes": ""
    })

dome_direct_calibration_summary_df = pd.DataFrame(calibration_rows)

dome_direct_calibration_summary_df.to_csv(
    OUTPUT_CALIBRATION_SUMMARY,
    index=False
)

print(f"Saved dome direct calibration summary to:\n{OUTPUT_CALIBRATION_SUMMARY}")

display(dome_direct_calibration_summary_df)

In [ ]:
def extract_waggle_runs(csv_file, row):
    """
    Extract one row per waggle run from an actual dome dance annotation file.
    """
    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    dance_id = strip_annotation_suffix(csv_file.name)

    date = str(row.get("date", "")).strip()
    bee_id = str(row.get("bee_id", "")).strip()
    time = str(row.get("time", "")).strip()
    condition = str(row.get("condition", "")).strip()

    aperture = row.get("aperture", "")
    notes = row.get("notes", "")

    source_batch = f"dome_direct_{date}"
    dance_key = f"{source_batch}__{bee_id}__{dance_id}"

    waggle_run_number = 0

    for session_index, annotator_row in df.iterrows():
        start_positions = parse_annotation_list(
            annotator_row["waggle_start_positions"],
            "waggle_start_positions"
        )

        start_frames = parse_annotation_list(
            annotator_row["waggle_start_frames"],
            "waggle_start_frames"
        )

        directions = parse_annotation_list(
            annotator_row["waggle_directions"],
            "waggle_directions"
        )

        if not (len(start_positions) == len(start_frames) == len(directions)):
            raise ValueError(
                f"{csv_file.name}: start positions, frames, and directions have different lengths."
            )

        for (start_x, start_y), start_frame, direction in zip(
            start_positions,
            start_frames,
            directions
        ):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            # Skip invalid 0,0 vectors
            if np.hypot(u, v) < 1e-8:
                continue

            waggle_run_number += 1

            video_angle_deg = vector_to_video_angle_deg(u, v)

            records.append({
                "source_batch": source_batch,
                "dance_key": dance_key,
                "dance_id": dance_id,
                "bee_id": bee_id,
                "date": date,
                "time": time,
                "condition": condition,
                "aperture": aperture,
                "notes": notes,
                "waggle_run": waggle_run_number,
                "csv_file": csv_file.name,
                "video_name": annotator_row["video_name"],
                "direction_u": u,
                "direction_v": v,
                "video_angle_deg": video_angle_deg,
                "start_x": start_x,
                "start_y": start_y,
                "start_xy": (start_x, start_y),
                "start_frame": start_frame,
                "annotation_session": session_index + 1,
                "calibration_file": row["calibration_file"],
                "calibration_id": row["calibration_id"],
                "orientation_method": "direct_compass"
            })

    return pd.DataFrame(records)


all_waggle_dfs = []
failed_rows = []

for idx, row in mapping_df.iterrows():
    try:
        dance_path = resolve_file(
            row["dance_file"],
            search_roots=[DOME_DANCE_ROOT],
            label="dance_file"
        )

        dance_df = extract_waggle_runs(dance_path, row)

        all_waggle_dfs.append(dance_df)

        print(f"Processed row {idx + 1}: {dance_path.name} ({len(dance_df)} waggle runs)")

    except Exception as error:
        print(f"\nCould not process row {idx + 1}")
        print(error)

        failed_rows.append({
            "row_number": idx + 1,
            "error": str(error),
            **row.to_dict()
        })

if len(all_waggle_dfs) == 0:
    raise ValueError("No dome direct waggle dances were processed successfully.")

dome_direct_waggle_df = pd.concat(
    all_waggle_dfs,
    ignore_index=True
)

if failed_rows:
    failed_df = pd.DataFrame(failed_rows)
    failed_df.to_csv(OUTPUT_FAILED_ROWS, index=False)
    print(f"\nSome rows failed. Saved failed row report to:\n{OUTPUT_FAILED_ROWS}")
else:
    print("\nAll mapped dome direct dances processed successfully.")

In [ ]:
dome_direct_waggle_df["calibration_id"] = dome_direct_waggle_df["calibration_id"].apply(
    clean_calibration_id
)

dome_direct_calibration_summary_df["calibration_id"] = dome_direct_calibration_summary_df[
    "calibration_id"
].apply(clean_calibration_id)

dome_direct_with_cal_df = dome_direct_waggle_df.merge(
    dome_direct_calibration_summary_df,
    on="calibration_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_from_calibration_summary")
)

# Use calibration_file from summary if needed, but keep original assignment too
if "calibration_file_from_calibration_summary" in dome_direct_with_cal_df.columns:
    dome_direct_with_cal_df = dome_direct_with_cal_df.rename(
        columns={
            "calibration_file": "assigned_calibration_file",
            "calibration_file_from_calibration_summary": "calibration_file"
        }
    )

dome_direct_with_cal_df["video_angle_deg"] = pd.to_numeric(
    dome_direct_with_cal_df["video_angle_deg"],
    errors="coerce"
)

dome_direct_with_cal_df["true_north_video_deg"] = pd.to_numeric(
    dome_direct_with_cal_df["true_north_video_deg"],
    errors="coerce"
)

dome_direct_with_cal_df["waggle_true_bearing_deg"] = (
    dome_direct_with_cal_df["video_angle_deg"]
    - dome_direct_with_cal_df["true_north_video_deg"]
) % 360

dome_direct_with_cal_df.loc[
    dome_direct_with_cal_df["true_north_video_deg"].isna(),
    "waggle_true_bearing_deg"
] = np.nan

# Check calibration matches
missing_cal = dome_direct_with_cal_df.loc[
    dome_direct_with_cal_df["calibration_id"].notna()
    & dome_direct_with_cal_df["true_north_video_deg"].isna(),
    ["assigned_calibration_file", "calibration_id"]
].drop_duplicates()

if len(missing_cal) == 0:
    print("All assigned calibrations matched successfully.")
else:
    print("These calibration assignments did not match the calibration summary:")
    display(missing_cal)

dome_direct_with_cal_df.to_csv(
    OUTPUT_WAGGLE_SUMMARY,
    index=False
)

print(f"\nSaved dome direct calibrated waggle runs to:\n{OUTPUT_WAGGLE_SUMMARY}")

display(
    dome_direct_with_cal_df[
        [
            "dance_key",
            "bee_id",
            "condition",
            "video_angle_deg",
            "assigned_calibration_file",
            "true_north_video_deg",
            "waggle_true_bearing_deg"
        ]
    ].head(20)
)

In [ ]:
print("Rows in final dome direct file:", len(dome_direct_with_cal_df))
print("Number of dances:", dome_direct_with_cal_df["dance_key"].nunique())
print("Number of bees:", dome_direct_with_cal_df["bee_id"].nunique())

print("\nRows with true_north_video_deg:")
print(dome_direct_with_cal_df["true_north_video_deg"].notna().value_counts(dropna=False))

print("\nCalibration annotation counts:")
display(
    dome_direct_calibration_summary_df[
        [
            "calibration_file",
            "calibration_n_annotations",
            "calibration_mean_north_deg",
            "calibration_circular_sd_deg",
            "true_north_video_deg"
        ]
    ].sort_values("calibration_file")
)

## Dome LANDMARK-TRANSFER calibrations

### North calibration for dome videos that do not have a suitable compass calibration video using the same annotated landmark in calibration video and waggle dance video

The goal is to turn this folder:

- one actual waggle dance annotation file

- one reference calibration file with compass north + landmark

- one landmark-only file from the waggle dance video

into a normal calibrated waggle-dance table containing:

- true_north_video_deg

- waggle_true_bearing_deg

- orientation_method = landmark_transfer

## Folder logic

For each folder, you have:

1. Actual waggle dance annotation

Example:

62b_antisolar_2025-01-21 11-37-17_waggle_annotations.csv

This contains the real waggle runs.

2. Reference calibration file

Example:

21_landmark_calibration_11-21-50_waggle_annotations.csv

This contains:

- 10 landmark annotations

3. Landmark annotation in the actual waggle dance video

Example:

21_landmark_11-37-17_waggle_annotations.csv

In [ ]:
# Videos that do not have a corresponding calibration file and need landmark-transfer
NEED_CALC_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome_north_calibrations\need_calc"
)

COMPASS_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome_north_calibrations\compass_files"
)

LANDMARK_MAPPING_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome_north_calibrations\dome_landmark_transfer_calibration\dome_landmark_transfer_MAPPING_FILLED.csv"
)

OUTPUT_WAGGLE_SUMMARY = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome_north_calibrations\dome_landmark_calibrated_waggle_runs.csv"
)

OUTPUT_ORIENTATION_SUMMARY = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\dome_north_calibrations\dome_landmark_orientation_summary.csv"
)

# West declination should be negative
DECLINATION_DEG = -21.27

In [ ]:
def parse_annotation_list(value, column_name=""):
    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value)

    value = re.sub(
        r"np\.float(?:16|32|64)?\(([^)]+)\)",
        r"\1",
        value
    )

    value = re.sub(r"\bnan\b", "None", value, flags=re.IGNORECASE)

    try:
        return list(ast.literal_eval(value))
    except (ValueError, SyntaxError) as error:
        raise ValueError(
            f"Could not parse column '{column_name}': {value}"
        ) from error


def vector_to_video_angle_deg(u, v):
    """
    Your video convention:
    0°   = top/up
    90°  = right
    180° = bottom/down
    270° = left
    """
    return np.degrees(np.arctan2(u, -v)) % 360


def circular_mean_deg(angles_deg):
    angles_rad = np.radians(angles_deg)
    mean_rad = np.arctan2(
        np.mean(np.sin(angles_rad)),
        np.mean(np.cos(angles_rad))
    )
    return np.degrees(mean_rad) % 360


def circular_resultant_length_deg(angles_deg):
    angles_rad = np.radians(angles_deg)
    return np.sqrt(
        np.mean(np.cos(angles_rad))**2
        + np.mean(np.sin(angles_rad))**2
    )


def circular_sd_deg(angles_deg):
    R = circular_resultant_length_deg(angles_deg)
    R = np.clip(R, 1e-12, 1)
    return np.degrees(np.sqrt(-2 * np.log(R)))


def strip_annotation_suffix(filename):
    name = Path(filename).name.strip()

    if name.endswith(".csv"):
        name = name[:-4]

    if name.endswith("_waggle_annotations"):
        name = name.replace("_waggle_annotations", "")

    return name


def resolve_file(file_value, search_roots, label="file"):
    """
    Resolve a file using priority search roots.

    Allows the mapping file to contain:
    - full path
    - relative path from one of the search roots
    - filename with .csv
    - filename without .csv

    Important:
    Searches roots in order and returns the first unambiguous match.
    This avoids false duplicate errors when the same file exists in several folders.
    """
    if pd.isna(file_value):
        raise ValueError(f"Missing {label} in mapping file.")

    value = str(file_value).strip().replace('"', "").replace("'", "")

    if value == "" or value.lower() == "nan":
        raise ValueError(f"Empty {label} in mapping file.")

    value_path = Path(value)

    # 1. Full path
    if value_path.exists():
        return value_path

    possible_values = [value]

    if not value.endswith(".csv"):
        possible_values.append(value + ".csv")

    # 2. Try exact relative path / direct filename inside each root, in priority order
    for root in search_roots:
        root = Path(root)

        for possible_value in possible_values:
            candidate = root / possible_value

            if candidate.exists():
                return candidate

    # 3. Recursive search, but root by root.
    # Return if one root gives exactly one match.
    all_matches = []

    for root in search_roots:
        root = Path(root)
        root_matches = []

        for possible_value in possible_values:
            possible_name = Path(possible_value).name
            root_matches.extend(root.rglob(possible_name))

        root_matches = sorted(set(root_matches))
        all_matches.extend(root_matches)

        if len(root_matches) == 1:
            return root_matches[0]

        if len(root_matches) > 1:
            raise ValueError(
                f"Found multiple matches for {label}: {value}\n"
                f"Within priority folder:\n{root}\n\n"
                "Use a relative path in the mapping file to disambiguate, for example:\n"
                r"20250121\22b_antisolar_2025-01-21 11-47-07_waggle_annotations\filename.csv"
                "\n\nMatches found:\n"
                + "\n".join(str(match) for match in root_matches)
            )

    raise FileNotFoundError(
        f"Could not find {label}: {value}\n"
        f"Searched in:\n"
        + "\n".join(str(root) for root in search_roots)
    )

In [ ]:
def extract_direction_angles_from_csv(csv_file):
    """
    For compass or landmark annotation files.
    Returns one row per annotation arrow.
    """
    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    for session_index, row in df.iterrows():
        positions = parse_annotation_list(
            row["waggle_start_positions"],
            "waggle_start_positions"
        )

        frames = parse_annotation_list(
            row["waggle_start_frames"],
            "waggle_start_frames"
        )

        directions = parse_annotation_list(
            row["waggle_directions"],
            "waggle_directions"
        )

        if not (len(positions) == len(frames) == len(directions)):
            raise ValueError(
                f"{csv_file.name}: positions, frames, and directions have different lengths."
            )

        for i, ((x, y), frame, direction) in enumerate(
            zip(positions, frames, directions),
            start=1
        ):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            if np.hypot(u, v) < 1e-8:
                continue

            video_angle_deg = vector_to_video_angle_deg(u, v)

            records.append({
                "source_file": csv_file.name,
                "annotation_number": len(records) + 1,
                "original_annotation_index": i,
                "frame": frame,
                "x": x,
                "y": y,
                "direction_u": u,
                "direction_v": v,
                "video_angle_deg": video_angle_deg,
                "annotation_session": session_index + 1
            })

    return pd.DataFrame(records)


def extract_waggle_runs(csv_file, source_batch):
    """
    For the actual dome waggle dance file.
    Returns one row per waggle run.
    """
    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    dance_id = strip_annotation_suffix(csv_file.name)
    dance_key = f"{source_batch}__{dance_id}"

    waggle_run_number = 0

    for session_index, row in df.iterrows():
        start_positions = parse_annotation_list(
            row["waggle_start_positions"],
            "waggle_start_positions"
        )

        start_frames = parse_annotation_list(
            row["waggle_start_frames"],
            "waggle_start_frames"
        )

        directions = parse_annotation_list(
            row["waggle_directions"],
            "waggle_directions"
        )

        if not (len(start_positions) == len(start_frames) == len(directions)):
            raise ValueError(
                f"{csv_file.name}: start positions, frames, and directions have different lengths."
            )

        for (start_x, start_y), start_frame, direction in zip(
            start_positions,
            start_frames,
            directions
        ):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            if np.hypot(u, v) < 1e-8:
                continue

            waggle_run_number += 1

            video_angle_deg = vector_to_video_angle_deg(u, v)

            records.append({
                "source_batch": source_batch,
                "dance_key": dance_key,
                "dance_id": dance_id,
                "waggle_run": waggle_run_number,
                "csv_file": csv_file.name,
                "video_name": row["video_name"],
                "direction_u": u,
                "direction_v": v,
                "video_angle_deg": video_angle_deg,
                "start_x": start_x,
                "start_y": start_y,
                "start_xy": (start_x, start_y),
                "start_frame": start_frame,
                "annotation_session": session_index + 1
            })

    return pd.DataFrame(records)

In [ ]:
def process_landmark_transfer_row(row):
    orientation_block = str(row["orientation_block"]).strip()

    # First resolve the actual waggle file globally under NEED_CALC_ROOT
    waggle_file = resolve_file(
        row["waggle_file"],
        search_roots=[NEED_CALC_ROOT],
        label="waggle_file"
    )

    # Useful folders based on the resolved waggle file
    waggle_folder = waggle_file.parent
    day_folder = waggle_folder.parent

    # Landmark from the actual dance video:
    # First look in the same folder as the waggle file.
    landmark_dance_file = resolve_file(
        row["landmark_dance_file"],
        search_roots=[waggle_folder, NEED_CALC_ROOT],
        label="landmark_dance_file"
    )

    # Reference landmark:
    # First look in the day folder, then same waggle folder, then globally.
    reference_landmark_file = resolve_file(
        row["reference_landmark_file"],
        search_roots=[day_folder, waggle_folder, NEED_CALC_ROOT],
        label="reference_landmark_file"
    )

    # Compass file stays in the separate compass folder
    reference_compass_file = resolve_file(
        row["reference_compass_file"],
        search_roots=[COMPASS_ROOT],
        label="reference_compass_file"
    )

    # Compass north from reference calibration video
    compass_df = extract_direction_angles_from_csv(reference_compass_file)

    if len(compass_df) == 0:
        raise ValueError(
            f"No valid compass annotations found in {reference_compass_file.name}"
        )

    magnetic_north_calibration_deg = circular_mean_deg(
        compass_df["video_angle_deg"].values
    )

    true_north_calibration_deg = (
        magnetic_north_calibration_deg - DECLINATION_DEG
    ) % 360

    compass_circular_sd = circular_sd_deg(
        compass_df["video_angle_deg"].values
    )

    # Landmark in the same reference calibration video
    reference_landmark_df = extract_direction_angles_from_csv(
        reference_landmark_file
    )

    if len(reference_landmark_df) == 0:
        raise ValueError(
            f"No valid landmark annotations found in {reference_landmark_file.name}"
        )

    landmark_calibration_deg = circular_mean_deg(
        reference_landmark_df["video_angle_deg"].values
    )

    landmark_calibration_circular_sd = circular_sd_deg(
        reference_landmark_df["video_angle_deg"].values
    )

    north_relative_to_landmark_deg = (
        true_north_calibration_deg - landmark_calibration_deg
    ) % 360

    # Landmark in actual waggle dance video
    dance_landmark_df = extract_direction_angles_from_csv(
        landmark_dance_file
    )

    if len(dance_landmark_df) == 0:
        raise ValueError(
            f"No valid landmark annotations found in {landmark_dance_file.name}"
        )

    landmark_dance_video_deg = circular_mean_deg(
        dance_landmark_df["video_angle_deg"].values
    )

    landmark_dance_circular_sd = circular_sd_deg(
        dance_landmark_df["video_angle_deg"].values
    )

    true_north_video_deg = (
        landmark_dance_video_deg + north_relative_to_landmark_deg
    ) % 360

    # Actual waggle runs
    waggle_df = extract_waggle_runs(
        waggle_file,
        source_batch=orientation_block
    )

    waggle_df["orientation_block"] = orientation_block
    waggle_df["bee_id"] = row.get("bee_id", "")
    waggle_df["condition"] = row.get("condition", "")
    waggle_df["orientation_notes"] = row.get("notes", "")

    waggle_df["orientation_method"] = "landmark_transfer"

    waggle_df["reference_compass_file"] = reference_compass_file.name
    waggle_df["reference_landmark_file"] = reference_landmark_file.name
    waggle_df["landmark_dance_file"] = landmark_dance_file.name

    waggle_df["magnetic_north_calibration_deg"] = magnetic_north_calibration_deg
    waggle_df["true_north_calibration_deg"] = true_north_calibration_deg
    waggle_df["compass_circular_sd_deg"] = compass_circular_sd

    waggle_df["landmark_calibration_deg"] = landmark_calibration_deg
    waggle_df["landmark_calibration_circular_sd_deg"] = landmark_calibration_circular_sd

    waggle_df["north_relative_to_landmark_deg"] = north_relative_to_landmark_deg

    waggle_df["landmark_dance_video_deg"] = landmark_dance_video_deg
    waggle_df["landmark_dance_circular_sd_deg"] = landmark_dance_circular_sd

    waggle_df["true_north_video_deg"] = true_north_video_deg

    waggle_df["waggle_true_bearing_deg"] = (
        waggle_df["video_angle_deg"] - waggle_df["true_north_video_deg"]
    ) % 360

    orientation_summary = {
        "orientation_block": orientation_block,
        "bee_id": row.get("bee_id", ""),
        "condition": row.get("condition", ""),
        "notes": row.get("notes", ""),

        "waggle_file": waggle_file.name,
        "landmark_dance_file": landmark_dance_file.name,
        "reference_landmark_file": reference_landmark_file.name,
        "reference_compass_file": reference_compass_file.name,

        "n_compass_annotations": len(compass_df),
        "magnetic_north_calibration_deg": magnetic_north_calibration_deg,
        "true_north_calibration_deg": true_north_calibration_deg,
        "compass_circular_sd_deg": compass_circular_sd,

        "n_reference_landmark_annotations": len(reference_landmark_df),
        "landmark_calibration_deg": landmark_calibration_deg,
        "landmark_calibration_circular_sd_deg": landmark_calibration_circular_sd,

        "north_relative_to_landmark_deg": north_relative_to_landmark_deg,

        "n_dance_landmark_annotations": len(dance_landmark_df),
        "landmark_dance_video_deg": landmark_dance_video_deg,
        "landmark_dance_circular_sd_deg": landmark_dance_circular_sd,

        "true_north_video_deg": true_north_video_deg,
        "n_waggle_runs": len(waggle_df),
        "orientation_method": "landmark_transfer"
    }

    return waggle_df, orientation_summary

In [ ]:
mapping_df = pd.read_csv(LANDMARK_MAPPING_FILE, dtype=str, encoding="utf-8-sig")
mapping_df.columns = mapping_df.columns.str.strip()

required_columns = [
    "orientation_block",
    "waggle_file",
    "landmark_dance_file",
    "reference_landmark_file",
    "bee_id",
    "reference_compass_file"
]

missing_columns = [
    col for col in required_columns
    if col not in mapping_df.columns
]

if missing_columns:
    raise ValueError(
        "Your mapping file is missing these columns:\n"
        + "\n".join(missing_columns)
    )

# Remove completely empty rows
mapping_df = mapping_df.replace(r"^\s*$", pd.NA, regex=True)
mapping_df = mapping_df.dropna(
    subset=required_columns,
    how="all"
).reset_index(drop=True)

all_waggle_dfs = []
orientation_rows = []
failed_rows = []

for idx, row in mapping_df.iterrows():
    try:
        waggle_df, orientation_summary = process_landmark_transfer_row(row)

        all_waggle_dfs.append(waggle_df)
        orientation_rows.append(orientation_summary)

        print(f"Processed row {idx + 1}: {orientation_summary['waggle_file']}")

    except Exception as error:
        print(f"\nCould not process row {idx + 1}")
        print(error)

        failed_rows.append({
            "row_number": idx + 1,
            "error": str(error),
            **row.to_dict()
        })

if len(all_waggle_dfs) == 0:
    raise ValueError("No landmark-transfer videos were processed successfully.")

dome_landmark_waggle_df = pd.concat(
    all_waggle_dfs,
    ignore_index=True
)

dome_landmark_orientation_summary_df = pd.DataFrame(
    orientation_rows
)

dome_landmark_waggle_df.to_csv(
    OUTPUT_WAGGLE_SUMMARY,
    index=False
)

dome_landmark_orientation_summary_df.to_csv(
    OUTPUT_ORIENTATION_SUMMARY,
    index=False
)

print(f"\nSaved calibrated landmark-transfer waggle runs to:\n{OUTPUT_WAGGLE_SUMMARY}")
print(f"\nSaved landmark-transfer orientation summary to:\n{OUTPUT_ORIENTATION_SUMMARY}")

if failed_rows:
    failed_df = pd.DataFrame(failed_rows)

    FAILED_ROWS_FILE = OUTPUT_ORIENTATION_SUMMARY.with_name(
        "dome_landmark_failed_rows.csv"
    )

    failed_df.to_csv(FAILED_ROWS_FILE, index=False)

    print(f"\nSome rows failed. Saved failed row report to:\n{FAILED_ROWS_FILE}")

In [ ]:
mapping_df = pd.read_csv(LANDMARK_MAPPING_FILE, dtype=str, encoding="utf-8-sig")

print("Mapping file path:")
print(LANDMARK_MAPPING_FILE)

print("\nColumns exactly as pandas reads them:")
for col in mapping_df.columns:
    print(repr(col))

print("\nFirst rows:")
display(mapping_df.head())

In [ ]:
display(
    dome_landmark_orientation_summary_df[
        [
            "orientation_block",
            "waggle_file",
            "reference_compass_file",
            "reference_landmark_file",
            "landmark_dance_file",
            "n_compass_annotations",
            "compass_circular_sd_deg",
            "n_reference_landmark_annotations",
            "landmark_calibration_circular_sd_deg",
            "n_dance_landmark_annotations",
            "landmark_dance_circular_sd_deg",
            "true_north_video_deg",
            "n_waggle_runs"
        ]
    ]
)

### How to read `dome_landmark_calibrated_waggle_runs.csv`

This file contains the dome waggle dances that were calibrated using landmark-transfer orientation correction.

Each row is one waggle run, not one dance. Therefore, the same dance-level and calibration-level values are repeated across all waggle runs belonging to the same dance.

### Important direction columns

- `direction_u`, `direction_v`  
  Raw unit direction vector from the waggle annotator.

- `video_angle_deg`  
  Waggle-run angle in raw video coordinates, calculated from `direction_u` and `direction_v`.

  Convention:
  - `0°` = top/up in the video
  - `90°` = right
  - `180°` = bottom/down
  - `270°` = left

- `true_north_video_deg`  
  Position of true geographic north in this specific dome video frame, after landmark-transfer calibration.

- `waggle_true_bearing_deg`  
  Final calibrated waggle-run bearing in geographic coordinates.

  Convention:
  - `0°` = true north
  - `90°` = east
  - `180°` = south
  - `270°` = west

  Calculated as:

  `waggle_true_bearing_deg = (video_angle_deg - true_north_video_deg) % 360`

This is the main angle column to use for geographic waggle-direction analysis.

| Column              | Meaning                                                                 |
| ------------------- | ----------------------------------------------------------------------- |
| `source_batch`      | Your orientation block / grouping label from the mapping file.          |
| `dance_key`         | Unique ID combining `source_batch` and `dance_id`. Useful for grouping. |
| `dance_id`          | Dance/video identifier without `_waggle_annotations.csv`.               |
| `waggle_run`        | Waggle run number within that dance.                                    |
| `csv_file`          | Actual waggle annotation CSV file.                                      |
| `video_name`        | Original video path stored by the annotator.                            |
| `bee_id`            | Bee ID from your mapping file.                                          |
| `condition`         | Condition from your mapping file, e.g. `solar` or `antisolar`.          |
| `orientation_notes` | Notes from your mapping file.                                           |

| Column               | Meaning                                                       |
| -------------------- | ------------------------------------------------------------- |
| `direction_u`        | x-component of annotator direction vector. Positive = right.  |
| `direction_v`        | y-component of annotator direction vector. Positive = down.   |
| `video_angle_deg`    | Raw video angle calculated from `direction_u`, `direction_v`. |
| `start_x`, `start_y` | Annotated start position of waggle run.                       |
| `start_xy`           | Same start position stored as tuple text.                     |
| `start_frame`        | Frame where the waggle run was annotated.                     |
| `annotation_session` | Row/session from the original annotator CSV. Usually 1.       |

| Column                                 | Meaning                                                                                          |
| -------------------------------------- | ------------------------------------------------------------------------------------------------ |
| `orientation_method`                   | Should be `landmark_transfer` for these rows.                                                    |
| `reference_compass_file`               | Compass north annotation file from the reference calibration video.                              |
| `reference_landmark_file`              | Landmark annotation file from the same reference calibration video.                              |
| `landmark_dance_file`                  | Landmark annotation file from the actual waggle dance video.                                     |
| `magnetic_north_calibration_deg`       | Mean compass-north direction in the reference calibration video, before declination correction.  |
| `true_north_calibration_deg`           | Compass north corrected for magnetic declination; true north in the reference calibration video. |
| `compass_circular_sd_deg`              | Circular SD of the compass annotations. Lower = more consistent.                                 |
| `landmark_calibration_deg`             | Mean landmark direction in the reference calibration video.                                      |
| `landmark_calibration_circular_sd_deg` | Circular SD of landmark annotations in the reference calibration video.                          |
| `north_relative_to_landmark_deg`       | Relationship between true north and the stable landmark.                                         |
| `landmark_dance_video_deg`             | Mean landmark direction in the actual waggle dance video.                                        |
| `landmark_dance_circular_sd_deg`       | Circular SD of the landmark annotations in the actual dance video.                               |
| `true_north_video_deg`                 | Reconstructed true north direction in the actual waggle dance video.                             |
| `waggle_true_bearing_deg`              | Final calibrated geographic waggle bearing.             